# Create Gradients Tasks

Steps to train a model on Gradients:

1. Run **Install Package** once.
2. Run **Setup** once and paste your Gradients API key when asked.
3. Pick one task section: Instruct, DPO, or Image.
4. Run that task cell.
5. The cell prints the task ID, current status, and trained model when it is ready.


## Install Package

Run this once at the top of the notebook. `%pip` installs the package into the same Python environment that this notebook is using.

In [ ]:
%pip install -q --upgrade gradientsio

Note: you may need to restart the kernel to use updated packages.


## Setup

Run this once after installing the package. It will ask for your API key and prepare the helper functions used by the task cells.

In [ ]:
import json
import os
from getpass import getpass

from gradientsio import GradientsClient
from gradientsio import TaskType


api_key = os.getenv("GRADIENTS_API_KEY") or getpass("Paste your Gradients API key: ").strip()
os.environ["GRADIENTS_API_KEY"] = api_key

client = GradientsClient(api_key=api_key)
created_tasks = {}

HOURS = 1
TEXT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
IMAGE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
WAIT_FOR_COMPLETION = False  # Change to True if you want the cell to wait until training finishes.
POLL_INTERVAL_SECONDS = 600


def as_dict(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json", exclude_none=True)
    if isinstance(value, dict):
        return {key: as_dict(item) for key, item in value.items()}
    if isinstance(value, list):
        return [as_dict(item) for item in value]
    return value


def print_json(value):
    print(json.dumps(as_dict(value), indent=2, sort_keys=True))


def show_task_result(task, task_name):
    created_tasks[task_name] = task.task_id
    print(f"Created {task_name} task: {task.task_id}")

    if WAIT_FOR_COMPLETION:
        details = task.wait(poll_interval=POLL_INTERVAL_SECONDS, raise_on_failure=False)
    else:
        details = task.refresh()

    print(f"Status: {details.status}")
    if details.trained_model_repository:
        print(f"Trained model: {details.trained_model_repository}")
    else:
        print("Training is not finished yet. Copy the task ID above and use the result checker at the bottom later.")

    return details


def check_task(task_id):
    task = client.tasks.handle(task_id)
    details = task.refresh()
    print(f"Status: {details.status}")
    if details.trained_model_repository:
        print(f"Trained model: {details.trained_model_repository}")
    else:
        print("No trained model yet. Check again later.")
    print_json(details)
    return details


print("Setup complete. Now run one task cell below.")

Setup complete. Now run one task cell below.


## Run Instruct Task

Run this cell to train on an instruction dataset.

In [3]:
task = client.train(
    model=TEXT_MODEL,
    task_type=TaskType.INSTRUCT,
    hours=HOURS,
    dataset="yahma/alpaca-cleaned",
    field_instruction="instruction",
    field_input="input",
    field_output="output",
)

instruct_result = show_task_result(task, "instruct")

NetworkError: The read operation timed out

## Run DPO Task

Run this cell to train from preference data with prompt, chosen, and rejected responses.

In [7]:
task = client.train(
    model=TEXT_MODEL,
    task_type=TaskType.DPO,
    hours=HOURS,
    dataset="trillionlabs/NemoSlides-DPO-mix-v1.0",
    field_prompt="prompt",
    field_chosen="chosen",
    field_rejected="rejected",
)

dpo_result = show_task_result(task, "dpo")

Created dpo task: eeaf00ef-2c35-41d7-a57b-cf9972114973
Status: pending
Training is not finished yet. Copy the task ID above and use the result checker at the bottom later.


## Run Image Task

Image LoRA training expects **one public or presigned zip URL**.

The zip should contain **10-50 image/caption pairs**. Each image must be a `.png`, `.jpg`, or `.jpeg` file, and each caption must be a `.txt` file with the same base name.

Example zip contents:

```text
0.png
0.txt
1.png
1.txt
2.jpg
2.txt
```

Each `.txt` file should contain the caption/prompt for the matching image. For example, `0.txt` is the caption for `0.png`.

In [ ]:
# Paste a public or presigned zip URL here.
# The zip should contain 10-50 LoRA training pairs with matching names:
# 0.png + 0.txt, 1.png + 1.txt, 2.jpg + 2.txt, etc.
IMAGE_DATASET_ZIP_URL = "https://example.com/my-image-lora-dataset.zip"

if "example.com" in IMAGE_DATASET_ZIP_URL:
    raise ValueError("Paste a real public or presigned image dataset zip URL into IMAGE_DATASET_ZIP_URL.")

task = client.tasks.create_image_zip(
    model_repo=IMAGE_MODEL,
    hours_to_complete=HOURS,
    model_type="sdxl",
    ds=IMAGE_DATASET_ZIP_URL,
)

image_result = show_task_result(task, "image")

AttributeError: 'TasksClient' object has no attribute 'create_image_zip'

## Check Results For Any Task ID

Use this later if training is still running. Paste the task ID printed by any task cell, then run this cell.

In [4]:
TASK_ID = "0b86a364-90d2-4ce4-aba9-f4c4169376fa"

if TASK_ID == "paste-task-id-here":
    print("Paste a task ID into TASK_ID first.")
else:
    result = check_task(TASK_ID)

Status: training
No trained model yet. Check again later.
{
  "account_id": "49ac55e1-b5e2-4c4d-883f-5edc112e2fcf",
  "base_model_repository": "Qwen/Qwen2.5-7B-Instruct",
  "created_at": "2026-05-13T19:35:47.657542Z",
  "ds_repo": "Hidden",
  "field_input": "input",
  "field_instruction": "instruct",
  "field_output": "output",
  "finished_at": "2026-05-13T20:35:47.657542Z",
  "hours_to_complete": 1.0,
  "id": "0b86a364-90d2-4ce4-aba9-f4c4169376fa",
  "status": "training",
  "task_type": "InstructTextTask"
}
